# Квантизация и дистилляция Qwen3

По запросу из MASSIVE ru-RU модель возвращает JSON с интентом и слотами.

## Окружение и данные

Выберите ядро `~/venvs/hse`. Данные и веса читаются с диска. Готовые прогоны ноутбук не перезаписывает: для нового запуска укажите другой каталог `ARTIFACTS`. Перед запуском проверьте номер свободной GPU в `CUDA_VISIBLE_DEVICES`.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

WORK = Path("hw7").resolve() if Path("hw7/experiment.py").is_file() else Path.cwd()
sys.path.insert(0, str(WORK))

from experiment import ARTIFACTS as SAVED_RUNS
from experiment import DATA, MODELS, compute_metrics, massive, read_jsonl

ARTIFACTS = SAVED_RUNS
env = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
env.setdefault("CUDA_VISIBLE_DEVICES", "5")

train, dev, intents, slot_types = massive(DATA)
print(f"Данные: {DATA} · модели: {MODELS} · артефакты: {ARTIFACTS}")
print(f"Обучение: {len(train)} запросов · проверка: {len(dev)} · GPU: {env['CUDA_VISIBLE_DEVICES']}")

## Бейзлайн и метрики

Все модели получают одну инструкцию. Генерация жадная, без рассуждений, до 128 новых токенов. Считаем качество на всех 2 033 запросах проверочной выборки.

Quality = ½ точности интента + ½ микро-F1 слотов. Ответ, не прошедший проверку JSON и схемы, считается ошибкой по интенту и фрейму; список предсказанных слотов считаем пустым. Тип слота `game_type` есть только в проверочной выборке: оставляем его в эталоне, но не даём модели в списке классов.

In [ ]:
def stage(mode, name, *options, required=()):
    folder = ARTIFACTS / name
    metrics = folder / "metrics/dev.json"
    if not (metrics.is_file() and all((folder / file).is_file() for file in required)):
        if folder.exists():
            raise FileExistsError(f"незаконченный прогон: {folder}; задайте новый ARTIFACTS")
        subprocess.run(
            [sys.executable, "-B", str(WORK / "experiment.py"), mode,
             "--run-name", name, "--artifacts", str(ARTIFACTS), *options],
            env=env, check=True,
        )
    predictions = list(read_jsonl(metrics.with_name("dev_predictions.jsonl")))
    if [row["id"] for row in predictions] != [row.id for row in dev]:
        raise ValueError(f"неполная проверочная выборка: {name}")
    scores = compute_metrics(predictions, intents, slot_types)
    if scores != json.loads(metrics.read_text(encoding="utf-8")):
        raise ValueError(f"пересчитанные метрики отличаются: {name}")
    return {"run": name, **scores}

def table(rows):
    return pd.DataFrame(rows).set_index("run").round(4)

In [ ]:
baseline = stage("baseline", "baseline_dense_8b", "--batch", "64")

## Квантизация

AWQ сравниваем на MASSIVE и WikiText-2: по 256 фрагментов по 64 токена, одна настройка весов. На MASSIVE меняем размер группы со 128 на 64. Для сравнения запускаем блочный GPTQ W4 и SmoothQuant W8A8 с `alpha=0.5`.

Чекпоинты хранят целочисленные веса. При проверке качества распаковываем их в BF16; скорость INT4/INT8 не измеряем.

In [ ]:
checkpoint = ("checkpoints/quantized/compression.json", "checkpoints/quantized/quantized.safetensors")
variants = [
    ("awq_massive_w4_g128", "awq", ()),
    ("awq_wikitext_w4_g128", "awq", ("--source", "wikitext")),
    ("awq_massive_w4_g64", "awq", ("--group-size", "64")),
    ("gptq_massive_w4_g128", "gptq", ()),
    ("smoothquant_massive_w8a8", "smoothquant", ()),
]
quant = [
    stage("quant", name, "--method", method, "--batch", "64", *opts, required=checkpoint)
    for name, method, opts in variants
]
display(table([baseline, *quant]))

## Дистилляция

Учитель Qwen3-32B отвечает на 4 000 обучающих запросов. Ученик Qwen3-8B учится только на валидных ответах учителя: токены инструкции, запроса и паддинга исключаем из потерь.

Сравниваем две скорости обучения на первой тысяче ответов и два объёма выборки при одинаковой скорости. Везде одна эпоха, AdamW, LoRA ранга 16, батч 8 и два шага накопления градиента. Цель — не меньше 90% качества учителя.

In [ ]:
teacher_name = "teacher_qwen3_32b"
teacher = stage(
    "teacher", teacher_name, "--batch", "32", "--samples", "4000",
    required=("checkpoints/teacher_answers/answers.jsonl", "metrics/train_answers.json"),
)
stats = json.loads((ARTIFACTS / teacher_name / "metrics/train_answers.json").read_text(encoding="utf-8"))
print(f"Валидные ответы учителя: {stats['valid']} из {stats['total']}")

In [ ]:
configs = [
    ("student_1000_lr1e4", "1000", "1e-4"),
    ("student_1000_lr2e4", "1000", "2e-4"),
    ("student_4000_lr2e4", "4000", "2e-4"),
]
students = [
    stage("student", name, "--batch", "64", "--samples", samples, "--lr", lr,
          required=("checkpoints/student_sft/config.json", "metrics/training.json"))
    for name, samples, lr in configs
]
display(table([baseline, teacher, *students]))

best = max(students, key=lambda run: run["quality"])
threshold = 0.9 * teacher["quality"]
print(f"Лучший ученик: {best['quality']:.2%} · порог: {threshold:.2%}")
print(ARTIFACTS / best["run"] / "checkpoints/student_sft")

## Файлы

В каталоге каждого прогона: параметры и версия кода, журнал, папки `metrics/` и `checkpoints/`. Ответы учителя лежат в `teacher_qwen3_32b/checkpoints/teacher_answers/answers.jsonl`. Чекпоинт ученика загружается через Transformers, квантизированный — через `load_quantized` из `quantize.py`.

Разбор результатов — в [отчёте](REPORT.md).